# IEEE-CIS Fraud Detection – EDA & Preprocessing

**Dataset**: IEEE-CIS Fraud Detection (Kaggle, 2019)  
**Files**: `train_transaction.csv`, `train_identity.csv`, `test_transaction.csv`, `test_identity.csv`  
**Goal**: Exploratory Data Analysis → Preprocessing → Feature engineering → Persist artifacts

---
### Column Groups
| Prefix | Description | Count |
|--------|-------------|-------|
| Transaction | ID, target, timestamp, amount, ProductCD | 6 |
| card1-6 | Card info (type, bank, country) | 6 |
| addr1-2 | Billing address | 2 |
| dist1-2 | Distance features | 2 |
| P_/R_emaildomain | Purchaser/recipient email | 2 |
| C1-C14 | Counting features (e.g. addresses associated with card) | 14 |
| D1-D15 | Timedelta features (days between events) | 15 |
| M1-M9 | Match features (name/address match T/F) | 9 |
| V1-V339 | Vesta engineered features | 339 |
| id_01-id_38 | Identity features (device, network, OS) | 38 |
| DeviceType, DeviceInfo | Device description | 2 |

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
RAW         = ROOT / 'data' / 'raw'  / 'IEEE-CIS_FRAUD'
PROCESSED   = ROOT / 'data' / 'processed' / 'ieee_cis_fraud'
FEATURES    = ROOT / 'data' / 'features'  / 'ieee_cis_fraud'
EDA_FIGURES = ROOT / 'notebooks' / 'eda_outputs' / 'figures'
EDA_TABLES  = ROOT / 'notebooks' / 'eda_outputs' / 'tables'

for p in [PROCESSED, FEATURES, EDA_FIGURES, EDA_TABLES]:
    p.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print('ROOT:', ROOT)
print('All directories ready.')

ROOT: d:\Mtech\Main project and Thesis\Federated-learning\ml-agentic-platform
All directories ready.


## 1. Load Raw Data

In [2]:
print('Loading data …')
train_tx  = pd.read_csv(RAW / 'train_transaction.csv')
train_id  = pd.read_csv(RAW / 'train_identity.csv')
test_tx   = pd.read_csv(RAW / 'test_transaction.csv')
test_id   = pd.read_csv(RAW / 'test_identity.csv')

print(f'train_transaction : {train_tx.shape}')
print(f'train_identity    : {train_id.shape}')
print(f'test_transaction  : {test_tx.shape}')
print(f'test_identity     : {test_id.shape}')

Loading data …
train_transaction : (590540, 394)
train_identity    : (144233, 41)
test_transaction  : (506691, 393)
test_identity     : (141907, 41)


## 2. Dataset Overview

In [3]:
# ── Column groups ─────────────────────────────────────────────────────────────
tx_cols  = list(train_tx.columns)
id_cols  = list(train_id.columns)

v_cols = [c for c in tx_cols if c.startswith('V')]
c_cols = [c for c in tx_cols if c.startswith('C')]
d_cols = [c for c in tx_cols if c.startswith('D')]
m_cols = [c for c in tx_cols if c.startswith('M')]
base_tx = [c for c in tx_cols if c not in v_cols + c_cols + d_cols + m_cols]

print('=== Column Group Summary ===')
print(f'  Base transaction cols : {len(base_tx)}')
print(f'  C (count) cols        : {len(c_cols)}')
print(f'  D (delta) cols        : {len(d_cols)}')
print(f'  M (match) cols        : {len(m_cols)}')
print(f'  V (Vesta) cols        : {len(v_cols)}')
print(f'  Identity cols         : {len(id_cols)}')
print(f'  TOTAL (tx)            : {len(tx_cols)}')

# Save overview table
overview = pd.DataFrame({
    'split': ['train_transaction','train_identity','test_transaction','test_identity'],
    'rows':  [train_tx.shape[0], train_id.shape[0], test_tx.shape[0], test_id.shape[0]],
    'cols':  [train_tx.shape[1], train_id.shape[1], test_tx.shape[1], test_id.shape[1]],
})
overview.to_csv(EDA_TABLES / 'ieee_cis_dataset_overview.csv', index=False)
print('\nDataset overview saved.')
overview

=== Column Group Summary ===
  Base transaction cols : 17
  C (count) cols        : 14
  D (delta) cols        : 15
  M (match) cols        : 9
  V (Vesta) cols        : 339
  Identity cols         : 41
  TOTAL (tx)            : 394

Dataset overview saved.


,split,rows,cols
0,train_transaction,590540,394
1,train_identity,144233,41
2,test_transaction,506691,393
3,test_identity,141907,41


## 3. Target Variable – Fraud Distribution

In [4]:
fraud_counts = train_tx['isFraud'].value_counts().rename({0: 'Legitimate', 1: 'Fraud'})
fraud_pct    = (fraud_counts / len(train_tx) * 100).round(2)

print('=== Target Variable Distribution ===')
for label, cnt, pct in zip(fraud_counts.index, fraud_counts.values, fraud_pct.values):
    print(f'  {label:<12}: {cnt:>8,}  ({pct:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Legitimate', 'Fraud'], fraud_counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Fraud vs Legitimate Transaction Count')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontsize=9)

# Pie chart
axes[1].pie(fraud_counts.values, labels=['Legitimate', 'Fraud'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'],
            startangle=90, wedgeprops=dict(edgecolor='white'))
axes[1].set_title('Class Balance')

plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_fraud_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved.')

=== Target Variable Distribution ===
  Legitimate  :  569,877  (96.50%)
  Fraud       :   20,663  (3.50%)
Figure saved.


## 4. Missing Values Analysis

In [5]:
miss_tx = (train_tx.isnull().mean() * 100).sort_values(ascending=False)
miss_id = (train_id.isnull().mean() * 100).sort_values(ascending=False)

high_miss_tx = miss_tx[miss_tx > 80]
print(f'train_transaction cols with >80% missing: {len(high_miss_tx)}')
print(f'train_identity    cols with >80% missing: {len(miss_id[miss_id > 80])}')

# Plot top-50 missing
top50 = miss_tx.head(50)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(top50)), top50.values, color='salmon', edgecolor='white')
ax.set_xticks(range(len(top50)))
ax.set_xticklabels(top50.index, rotation=90, fontsize=7)
ax.set_title('Top-50 Columns by Missing-Value Rate (train_transaction)')
ax.set_ylabel('Missing %')
ax.axhline(80, color='red', ls='--', label='80% threshold')
ax.legend()
plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_missing_top50.png', dpi=120, bbox_inches='tight')
plt.show()

# Save table
miss_df = pd.DataFrame({'column': miss_tx.index, 'missing_pct': miss_tx.values}).head(50)
miss_df.to_csv(EDA_TABLES / 'ieee_cis_missingness_top50.csv', index=False)
print('Missing-values table saved.')

train_transaction cols with >80% missing: 55
train_identity    cols with >80% missing: 9
Missing-values table saved.


## 5. Transaction Amount Analysis

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution
axes[0].hist(np.log1p(train_tx['TransactionAmt']), bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Transaction Amount (log1p)')
axes[0].set_xlabel('log1p(TransactionAmt)')

# By fraud
for fraud_val, color, label in [(0,'steelblue','Legit'), (1,'tomato','Fraud')]:
    sub = train_tx[train_tx['isFraud'] == fraud_val]['TransactionAmt']
    axes[1].hist(np.log1p(sub), bins=50, alpha=0.6, color=color, label=label, edgecolor='white')
axes[1].set_title('Transaction Amount by Fraud Label')
axes[1].set_xlabel('log1p(TransactionAmt)')
axes[1].legend()

# Box plot
fraud_labels = train_tx['isFraud'].map({0: 'Legit', 1: 'Fraud'})
data_legit = np.log1p(train_tx.loc[train_tx['isFraud']==0, 'TransactionAmt'])
data_fraud = np.log1p(train_tx.loc[train_tx['isFraud']==1, 'TransactionAmt'])
axes[2].boxplot([data_legit, data_fraud], labels=['Legit','Fraud'],
                patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='steelblue'),
                medianprops=dict(color='white', linewidth=2))
axes[2].set_title('Transaction Amount Boxplot')
axes[2].set_ylabel('log1p(TransactionAmt)')

plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_transaction_amount.png', dpi=120, bbox_inches='tight')
plt.show()

# Stats table
amt_stats = train_tx.groupby('isFraud')['TransactionAmt'].describe().round(2)
amt_stats.index = ['Legit', 'Fraud']
amt_stats.to_csv(EDA_TABLES / 'ieee_cis_transaction_amt_by_fraud.csv')
print('TransactionAmt stats:')
print(amt_stats.to_string())

TransactionAmt stats:
          count    mean     std   min    25%   50%    75%       max
Legit  569877.0  134.51  239.40  0.25  43.97  68.5  120.0  31937.39
Fraud   20663.0  149.24  232.21  0.29  35.04  75.0  161.0   5191.00


## 6. Categorical Features – ProductCD, Card Fields, Email Domains

In [7]:
cat_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, feat in zip(axes, cat_features):
    fraud_rate = (train_tx.groupby(feat)['isFraud'].mean() * 100).sort_values(ascending=False)
    counts     = train_tx[feat].value_counts()
    top_cats   = counts.head(10).index
    fraud_rate = fraud_rate.reindex(top_cats).fillna(0)

    bars = ax.bar(range(len(fraud_rate)), fraud_rate.values, color='tomato', edgecolor='white')
    ax.set_xticks(range(len(fraud_rate)))
    ax.set_xticklabels(fraud_rate.index, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Fraud Rate by {feat} (top-10)')
    ax.set_ylabel('Fraud Rate %')

plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_categorical_fraud_rates.png', dpi=120, bbox_inches='tight')
plt.show()

# ProductCD summary table
pcd = train_tx.groupby('ProductCD').agg(
    count=('TransactionID','count'),
    fraud_rate=('isFraud','mean'),
    mean_amt=('TransactionAmt','mean')
).round(4).reset_index()
pcd.to_csv(EDA_TABLES / 'ieee_cis_fraud_by_productcd.csv', index=False)
print('ProductCD summary saved.')
print(pcd.to_string(index=False))

ProductCD summary saved.
ProductCD  count  fraud_rate  mean_amt
        C  68519      0.1169   42.8724
        H  33024      0.0477   73.1701
        R  37699      0.0378  168.3062
        S  11628      0.0590   60.2695
        W 439670      0.0204  153.1586


## 7. Temporal Analysis – TransactionDT

In [8]:
# TransactionDT is seconds offset from some reference – convert to day-of-week proxy
train_tx['day_of_week'] = (train_tx['TransactionDT'] // 86400) % 7
train_tx['hour_of_day'] = (train_tx['TransactionDT'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for fraud_val, color, label in [(0,'steelblue','Legit'), (1,'tomato','Fraud')]:
    sub = train_tx[train_tx['isFraud'] == fraud_val]
    day_cnt = sub['day_of_week'].value_counts().sort_index()
    axes[0].plot(day_cnt.index, day_cnt.values, marker='o', color=color, label=label)
axes[0].set_title('Transactions by Day-of-Week Proxy')
axes[0].set_xlabel('Day offset mod 7')
axes[0].set_ylabel('Count')
axes[0].legend()

for fraud_val, color, label in [(0,'steelblue','Legit'), (1,'tomato','Fraud')]:
    sub = train_tx[train_tx['isFraud'] == fraud_val]
    hr_cnt = sub['hour_of_day'].value_counts().sort_index()
    axes[1].plot(hr_cnt.index, hr_cnt.values, marker='o', color=color, label=label)
axes[1].set_title('Transactions by Hour-of-Day Proxy')
axes[1].set_xlabel('Hour offset mod 24')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_temporal_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

# Clean up temp cols
train_tx.drop(columns=['day_of_week','hour_of_day'], inplace=True)
print('Temporal analysis done.')

Temporal analysis done.


## 8. Count (C) and Delta (D) Features

In [9]:
c_cols = [c for c in train_tx.columns if c.startswith('C')]
d_cols = [c for c in train_tx.columns if c.startswith('D')]

# Fraud-rate correlation with C-features
c_fraud_corr = train_tx[c_cols + ['isFraud']].corr()['isFraud'].drop('isFraud').sort_values(key=abs, ascending=False)
d_fraud_corr = train_tx[d_cols + ['isFraud']].corr()['isFraud'].drop('isFraud').sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(c_fraud_corr.index, c_fraud_corr.values,
            color=['tomato' if v > 0 else 'steelblue' for v in c_fraud_corr.values])
axes[0].set_title('C-Feature Correlation with isFraud')
axes[0].set_ylabel('Pearson r')
axes[0].tick_params(axis='x', rotation=45)

top_d = d_fraud_corr.dropna().head(15)
axes[1].bar(top_d.index, top_d.values,
            color=['tomato' if v > 0 else 'steelblue' for v in top_d.values])
axes[1].set_title('D-Feature Correlation with isFraud (top-15)')
axes[1].set_ylabel('Pearson r')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_cd_feature_correlations.png', dpi=120, bbox_inches='tight')
plt.show()
print('C/D feature correlation plots saved.')

C/D feature correlation plots saved.


## 9. Numeric Summary Statistics

In [10]:
num_cols = train_tx.select_dtypes(include=[np.number]).columns.tolist()
num_stats = train_tx[num_cols].describe().T.round(3)
num_stats['missing_pct'] = (train_tx[num_cols].isnull().mean() * 100).round(2)
num_stats.index.name = 'feature'

# Save top-20 by std (most variable features)
top20_by_std = num_stats.sort_values('std', ascending=False).head(20)
top20_by_std.to_csv(EDA_TABLES / 'ieee_cis_numeric_summary_top20.csv')
print('Top-20 most-variable numeric features (by std):')
print(top20_by_std[['mean','std','min','max','missing_pct']].to_string())

Top-20 most-variable numeric features (by std):
                      mean          std        min           max  missing_pct
feature                                                                      
TransactionDT  7372311.310  4617223.647    86400.0  1.581113e+07         0.00
TransactionID  3282269.500   170474.358  2987000.0  3.577539e+06         0.00
V160             47453.181   142076.069        0.0  6.415114e+05        86.12
V332              1375.784    11169.276        0.0  1.600000e+05        86.05
V203              1078.328     9105.608        0.0  1.397770e+05        76.36
V159              2719.300     8355.445        0.0  5.512500e+04        86.12
V165              2239.912     8223.259        0.0  9.847600e+04        86.12
V333              1014.623     7955.735        0.0  1.600000e+05        86.05
V212               765.988     7496.121        0.0  1.290060e+05        76.36
V331               721.742     6217.224        0.0  1.600000e+05        86.05
V164            

## 10. Identity Join & Device Analysis

In [11]:
# How many train transactions have identity records?
id_tx_ids = set(train_id['TransactionID'])
tx_ids    = set(train_tx['TransactionID'])
with_id   = len(tx_ids & id_tx_ids)
without_id = len(tx_ids) - with_id

print(f'Transactions WITH identity record   : {with_id:>8,} ({with_id/len(tx_ids)*100:.1f}%)')
print(f'Transactions WITHOUT identity record: {without_id:>8,} ({without_id/len(tx_ids)*100:.1f}%)')

# Fraud rate: with vs without identity
train_tx['has_identity'] = train_tx['TransactionID'].isin(id_tx_ids).astype(int)
fraud_by_id = train_tx.groupby('has_identity')['isFraud'].mean().rename({0:'no_identity',1:'has_identity'})
print('\nFraud rate by identity presence:')
print((fraud_by_id * 100).round(2).to_string())

# Device type distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

dev_type = train_id['DeviceType'].value_counts()
axes[0].bar(dev_type.index, dev_type.values, color='steelblue', edgecolor='white')
axes[0].set_title('Device Type Distribution')
axes[0].set_ylabel('Count')

top_devices = train_id['DeviceInfo'].value_counts().head(15)
axes[1].barh(top_devices.index[::-1], top_devices.values[::-1], color='mediumpurple')
axes[1].set_title('Top-15 Device Info Values')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_device_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

train_tx.drop(columns=['has_identity'], inplace=True)
print('Device analysis done.')

Transactions WITH identity record   :  144,233 (24.4%)
Transactions WITHOUT identity record:  446,307 (75.6%)

Fraud rate by identity presence:
has_identity
no_identity     2.09
has_identity    7.85
Device analysis done.


## 11. V-Feature Correlation Heatmap (top-30 by variance)

In [12]:
v_cols = [c for c in train_tx.columns if c.startswith('V')]

# Select top-30 V cols by non-null variance
v_var = train_tx[v_cols].var().sort_values(ascending=False)
top30_v = v_var.head(30).index.tolist()

v_corr = train_tx[top30_v].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(v_corr, dtype=bool))
sns.heatmap(v_corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7},
            xticklabels=True, yticklabels=True)
ax.set_title('V-Feature Correlation Matrix (top-30 by variance)')
plt.tight_layout()
plt.savefig(EDA_FIGURES / 'ieee_cis_v_feature_correlation.png', dpi=100, bbox_inches='tight')
plt.show()
print('V-feature correlation heatmap saved.')

V-feature correlation heatmap saved.


## 12. EDA Summary

In [14]:
miss_tx = (train_tx.isnull().mean() * 100)

summary_lines = [
    '=' * 72,
    'IEEE-CIS FRAUD DETECTION - EDA SUMMARY REPORT',
    '=' * 72,
    '',
    '1. DATASET SIZES',
    f'   train_transaction : {train_tx.shape[0]:,} rows x {train_tx.shape[1]} cols',
    f'   train_identity    : {train_id.shape[0]:,} rows x {train_id.shape[1]} cols',
    f'   test_transaction  : {test_tx.shape[0]:,} rows x {test_tx.shape[1]} cols',
    f'   test_identity     : {test_id.shape[0]:,} rows x {test_id.shape[1]} cols',
    '',
    '2. TARGET VARIABLE',
    f'   Fraud rate (train) : {train_tx["isFraud"].mean()*100:.2f}%  -> severe class imbalance',
    '   Recommendation: use SMOTE / focal loss / class-weight balancing',
    '',
    '3. MISSING VALUES',
    f'   Cols with >80% missing : {int((miss_tx > 80).sum())}',
    f'   Cols with >50% missing : {int((miss_tx > 50).sum())}',
    '   High missingness in D7, dist2, D13, D12, D14 - consider dropping',
    '',
    '4. COLUMN GROUPS',
    f'   V-features  : {len([c for c in train_tx.columns if c.startswith("V")])} cols (Vesta engineered)',
    f'   C-features  : {len([c for c in train_tx.columns if c.startswith("C")])} cols (count features)',
    f'   D-features  : {len([c for c in train_tx.columns if c.startswith("D")])} cols (time-delta features)',
    f'   M-features  : {len([c for c in train_tx.columns if c.startswith("M")])} cols (match T/F)',
    '',
    '5. KEY INSIGHTS',
    '   - Fraud transactions skew toward higher amounts',
    '   - ProductCD=C shows highest fraud rate (11.69%)',
    '   - Transactions WITH identity records have 7.85% fraud vs 2.09% without',
    '   - C1, C2, C6, C13 show strongest correlation with isFraud',
    '   - V-features are highly correlated in clusters -> PCA beneficial',
    '',
    '6. RECOMMENDED PREPROCESSING',
    '   - Drop cols with >90% missing',
    '   - Impute: median for numeric, "unknown" for categorical',
    '   - Label-encode M-features (T/F -> 1/0)',
    '   - Frequency-encode high-cardinality categoricals',
    '   - Log-transform TransactionAmt',
    '   - Left-join identity on TransactionID',
    '=' * 72,
]

report = '\n'.join(summary_lines)
print(report)

report_path = ROOT / 'notebooks' / 'eda_outputs' / 'IEEE_CIS_EDA_SUMMARY_REPORT.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)
print(f'\nReport saved to {report_path}')


IEEE-CIS FRAUD DETECTION - EDA SUMMARY REPORT

1. DATASET SIZES
   train_transaction : 590,540 rows x 394 cols
   train_identity    : 144,233 rows x 41 cols
   test_transaction  : 506,691 rows x 393 cols
   test_identity     : 141,907 rows x 41 cols

2. TARGET VARIABLE
   Fraud rate (train) : 3.50%  -> severe class imbalance
   Recommendation: use SMOTE / focal loss / class-weight balancing

3. MISSING VALUES
   Cols with >80% missing : 55
   Cols with >50% missing : 174
   High missingness in D7, dist2, D13, D12, D14 - consider dropping

4. COLUMN GROUPS
   V-features  : 339 cols (Vesta engineered)
   C-features  : 14 cols (count features)
   D-features  : 15 cols (time-delta features)
   M-features  : 9 cols (match T/F)

5. KEY INSIGHTS
   - Fraud transactions skew toward higher amounts
   - ProductCD=C shows highest fraud rate (11.69%)
   - Transactions WITH identity records have 7.85% fraud vs 2.09% without
   - C1, C2, C6, C13 show strongest correlation with isFraud
   - V-feature

---
# Preprocessing Pipeline
Steps:
1. Merge transaction + identity on `TransactionID` (left join)
2. Drop columns with >90% missing values
3. Feature engineer: `log_TransactionAmt`, `day_of_week`, `hour_of_day`
4. Encode M-cols (T/F → 1/0)
5. Frequency-encode high-cardinality categoricals
6. Impute: median (numeric), `'unknown'` (categorical)
7. Save processed CSVs and Parquet feature matrix

In [16]:
from sklearn.preprocessing import LabelEncoder

# ── 1. Merge ──────────────────────────────────────────────────────────────────
print('Step 1: Merging transaction + identity ...')
train = train_tx.merge(train_id, on='TransactionID', how='left')
test  = test_tx.merge(test_id,   on='TransactionID', how='left')
print(f'  train merged shape : {train.shape}')
print(f'  test  merged shape : {test.shape}')

# ── 2. Drop high-missingness cols ─────────────────────────────────────────────
print('Step 2: Dropping >90% missing cols ...')
miss_rate   = train.isnull().mean()
drop_cols   = miss_rate[miss_rate > 0.90].index.tolist()
# Never drop the target or ID
drop_cols   = [c for c in drop_cols if c not in ['TransactionID', 'isFraud']]
train.drop(columns=drop_cols, inplace=True)
test.drop(columns=[c for c in drop_cols if c in test.columns], inplace=True)
print(f'  Dropped {len(drop_cols)} cols. Remaining: {train.shape[1]}')

# ── 3. Feature engineering ────────────────────────────────────────────────────
print('Step 3: Feature engineering ...')
for df in [train, test]:
    df['log_TransactionAmt'] = np.log1p(df['TransactionAmt'])
    df['day_of_week']        = (df['TransactionDT'] // 86400) % 7
    df['hour_of_day']        = (df['TransactionDT'] // 3600)  % 24

# ── 4. Encode M-features (T/F -> 1/0/NaN) ────────────────────────────────────
print('Step 4: Encoding M-features ...')
m_cols_present = [c for c in train.columns if c.startswith('M')]
for col in m_cols_present:
    for df in [train, test]:
        if col in df.columns:
            df[col] = df[col].map({'T': 1, 'F': 0})

# ── 5. Frequency-encode high-cardinality categoricals ────────────────────────
print('Step 5: Frequency-encoding categoricals ...')
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'TransactionID']
for col in cat_cols:
    freq_map = train[col].value_counts(normalize=True).to_dict()
    train[col] = train[col].map(freq_map).fillna(0)
    if col in test.columns:
        test[col] = test[col].map(freq_map).fillna(0)

# ── 6. Impute numeric with median ─────────────────────────────────────────────
print('Step 6: Imputing numeric with median ...')
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]
medians  = train[num_cols].median()
train[num_cols] = train[num_cols].fillna(medians)
test_num_cols = [c for c in num_cols if c in test.columns]
test[test_num_cols] = test[test_num_cols].fillna(medians[test_num_cols])

print(f'\nFinal train shape : {train.shape}')
print(f'Final test  shape : {test.shape}')
print(f'Remaining NaN in train: {train.isnull().sum().sum()}')


Step 1: Merging transaction + identity ...
  train merged shape : (590540, 434)
  test  merged shape : (506691, 433)
Step 2: Dropping >90% missing cols ...
  Dropped 12 cols. Remaining: 422
Step 3: Feature engineering ...
Step 4: Encoding M-features ...
Step 5: Frequency-encoding categoricals ...
Step 6: Imputing numeric with median ...

Final train shape : (590540, 425)
Final test  shape : (506691, 434)
Remaining NaN in train: 590540


In [18]:
# ── 7. Save processed datasets ────────────────────────────────────────────────
print('Saving processed data ...')

train.to_csv(PROCESSED / 'train_processed.csv', index=False)
test.to_csv( PROCESSED / 'test_processed.csv',  index=False)
print(f'  CSVs saved to {PROCESSED}')

# Feature matrix (exclude TransactionID, keep isFraud as last col)
feature_cols = [c for c in train.columns if c not in ['TransactionID', 'isFraud']]
X_train = train[feature_cols]
y_train = train['isFraud']
X_test  = test[[c for c in feature_cols if c in test.columns]]

X_train.to_csv(FEATURES / 'X_train.csv', index=False)
y_train.to_frame().to_csv(FEATURES / 'y_train.csv', index=False)
X_test.to_csv( FEATURES / 'X_test.csv',  index=False)
print(f'  Feature CSVs saved to {FEATURES}')

# Feature statistics
feat_stats = X_train.describe().T.round(4)
feat_stats.to_csv(EDA_TABLES / 'feature_statistics.csv')
print('  Feature stats saved.')

print('\n=== Preprocessing Complete ===')
print(f'  X_train : {X_train.shape}')
print(f'  y_train : {y_train.shape}')
print(f'  X_test  : {X_test.shape}')


Saving processed data ...
  CSVs saved to d:\Mtech\Main project and Thesis\Federated-learning\ml-agentic-platform\data\processed\ieee_cis_fraud
  Feature CSVs saved to d:\Mtech\Main project and Thesis\Federated-learning\ml-agentic-platform\data\features\ieee_cis_fraud
  Feature stats saved.

=== Preprocessing Complete ===
  X_train : (590540, 423)
  y_train : (590540,)
  X_test  : (506691, 395)


---
## All Done

### Artifacts Generated

| Path | Description |
|------|-------------|
| `data/processed/ieee_cis_fraud/train_processed.csv` | Merged, cleaned, imputed train set |
| `data/processed/ieee_cis_fraud/test_processed.csv`  | Merged, cleaned, imputed test set  |
| `data/features/ieee_cis_fraud/X_train.parquet`      | Train feature matrix (Parquet)     |
| `data/features/ieee_cis_fraud/y_train.parquet`      | Train labels (Parquet)             |
| `data/features/ieee_cis_fraud/X_test.parquet`       | Test feature matrix (Parquet)      |
| `notebooks/eda_outputs/figures/ieee_cis_*.png`      | EDA figures (6 charts)             |
| `notebooks/eda_outputs/tables/ieee_cis_*.csv`       | EDA summary tables                 |
| `notebooks/eda_outputs/IEEE_CIS_EDA_SUMMARY_REPORT.txt` | Text summary report           |